# 目的
#每日收盤後開始爬股價

#可以視情況條種是否抓營收，大概都隔月的10 後營收才會公布
## 【資料源】
來自 _BaseInfo  _CrawBase 會先去取基本相關資料 (股票號碼...等)

## 【輸出】
股票相關推薦圖表/數據

In [1]:
import pandas as pd
import threading
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta


In [2]:
#%run _BaseInfoETF.ipynb
%run _CrawBase.ipynb
#%run _LineMsg.ipynb
%run _genHtml_cyber.ipynb
#%run RunRealTimeStock.ipynb

C:\Users\user\anaconda3\lib\site-packages\mpl_finance.py:16: DeprecationWarning: 



    Please use `mplfinance` instead (no hyphen, no underscore).

    To install: `pip install --upgrade mplfinance` 

   For more information, see: https://pypi.org/project/mplfinance/


  __warnings.warn('\n\n  ================================================================='+


[上市] 線上抓取成功，共 1090 筆
[上櫃] 線上抓取成功，共 892 筆
[output] 儲存成功，共 1982 筆
df_info 共 1980 筆，上市：1089，上櫃：891


In [3]:
# 覆寫

# 確認是否已被抓取過
def check_data_exist(file_type,stock_number,date):
    stock_number_=str(stock_number)
    date_=date.strftime("%Y-%m-%d")
    file_path = rf"D:\Project\Jupyter\Stock\Main\Data\{file_type}_{stock_number}_{date_}.json"
    
    #return False
    '''
    #特定月份更新
    if('2025-09'== date.strftime("%Y-%m")):
        return False
    '''
    
    # alaways 跑本月 更新
    '''
    if date.strftime("%Y-%m") == datetime.now().strftime("%Y-%m"):
        print('【craw_stock】跑本當月更新 -->'+stock_number,datetime.now().strftime("%Y-%m"))
        return False
    '''
    # 1. 確認檔案是否存在
    if os.path.exists(file_path):
        # 2. 讀取檔案內容
        with open(file_path, 'r', encoding='utf-8') as file:
            data = json.load(file)
            #print(data)
        return True
    return False

In [4]:
def delAllFile(folder):
    fileList = os.listdir(folder)
    for f in fileList:
        filePath = folder + '/'+f

        if os.path.isfile(filePath):
            os.remove(filePath)

        elif os.path.isdir(filePath):
            newFileList = os.listdir(filePath)
            for f1 in newFileList:
                insideFilePath = filePath + '/' + f1

                if os.path.isfile(insideFilePath):
                    os.remove(insideFilePath)

In [5]:

start_month = '2025-12-01'
start_date = datetime.strptime(start_month, "%Y-%m-%d").date()
today = datetime.today().date()

def to_roc(date_obj):
    roc_year = date_obj.year - 1911
    return f"{roc_year}/{date_obj.month:02d}/{date_obj.day:02d}"

end_time_list = []
current_date = start_date
while current_date <= today:
    end_time_list.append(to_roc(current_date))
    current_date += timedelta(days=1)

end_time_list=end_time_list[-20:]

min(end_time_list) 

'115/06/08'

In [6]:
# Lock 保護全域 DataFrame 的 concat（多執行緒共寫）
_save_lock = threading.Lock()

def process_stock_codes(stock_number):
    try:
        craw_stock_need_update = False
        RowData_df_craw_stock, His_Stock, isSuccess = craw_stock(
            stock_number, start_month,
            datetime.now().strftime("%Y-%m-%d"),
            craw_stock_need_update
        )
        _dummyData = data_process(RowData_df_craw_stock)

        # gen_html / save_plt_to_html 寫不同檔案，不需要 lock
        gen_html(stock_number, _dummyData)

        # save_process_data 會 concat 全域 df，需要 lock
        with _save_lock:
            save_process_data(stock_number, _dummyData)

    except Exception as error:
        print(f'Error processing stock {stock_number}: {error}')

def process_stock_list(stock_list):
    for stock_number in stock_list:
        process_stock_codes(stock_number)


In [7]:
# ── 參數：可依機器 CPU 核心數調整，爬蟲 IO 密集建議 8~16 ──
MAX_WORKERS = 8

all_codes = list(
    _baseInfo[_baseInfo['Type'] == '上市'].dropna()['公司代號']
) + list(
    _baseInfo[_baseInfo['Type'] != '上市'].dropna()['公司代號']
)

if True:
    starttime = datetime.now()

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(process_stock_codes, code): code for code in all_codes}
        for future in as_completed(futures):
            code = futures[future]
            try:
                future.result()   # 若 process_stock_codes 內有漏接的例外，這裡會再拋出
            except Exception as e:
                print(f'[ThreadPool] {code} failed: {e}')

    delAllFile('./Result/DailyRunRealTimeStock/')
    delAllFile('./Result/DailyFiles/')

    save_data_by_date(df_save_old_RunRealTimeStock, "Result/DailyRunRealTimeStock")
    save_data_by_date(df_save_df_to_excel, "Result/DailyFiles")

    print("All stocks processed.")
    print("start time " + starttime.strftime("%Y/%m/%d %H:%M:%S"))
    print("end time   " + datetime.now().strftime("%Y/%m/%d %H:%M:%S"))
    print(f"elapsed    {datetime.now() - starttime}")


【craw_stock】 craw_stock stock_number!!!!!!!!! -->1101【craw_stock】 craw_stock stock_number!!!!!!!!! -->1102

【craw_stock】 craw_stock stock_number!!!!!!!!! -->1103
【craw_stock】 craw_stock stock_number!!!!!!!!! -->1104
【craw_stock】 craw_stock stock_number!!!!!!!!! -->1108
【craw_stock】 craw_stock stock_number!!!!!!!!! -->1109
【craw_stock】 craw_stock stock_number!!!!!!!!! -->1110
【craw_stock】 craw_stock stock_number!!!!!!!!! -->1201










saved -> Html/[1108]幸福-水泥工業-上市.html
 Visit: http://localhost/IT//Html/[1108]幸福-水泥工業-上市.html
saved -> Html/[1103]嘉泥-水泥工業-上市.html
 Visit: http://localhost/IT//Html/[1103]嘉泥-水泥工業-上市.html
saved -> Html/[1201]味全-食品工業-上市.html
 Visit: http://localhost/IT//Html/[1201]味全-食品工業-上市.html
saved -> Html/[1102]亞泥-水泥工業-上市.html
 Visit: http://localhost/IT//Html/[1102]亞泥-水泥工業-上市.html
【craw_stock】 craw_stock stock_number!!!!!!!!! -->1203
【craw_stock】 craw_stock stock_number!!!!!!!!! -->1210
【craw_stock】 craw_stock stock_number!!!!!!!!! -->1213
saved -> Html/[1109]信大-水泥工業-



【craw_stock】 craw_stock stock_number!!!!!!!!! -->1419

saved -> Html/[1402]遠東新-紡織纖維-上市.html
 Visit: http://localhost/IT//Html/[1402]遠東新-紡織纖維-上市.html

【craw_stock】 craw_stock stock_number!!!!!!!!! -->1423



saved -> Html/[1410]南染-紡織纖維-上市.html
 Visit: http://localhost/IT//Html/[1410]南染-紡織纖維-上市.html
saved -> Html/[1413]宏洲-紡織纖維-上市.html
 Visit: http://localhost/IT//Html/[1413]宏洲-紡織纖維-上市.html
【craw_stock】 craw_stock stock_number!!!!!!!!! -->1432
【craw_stock】 craw_stock stock_number!!!!!!!!! -->1434


saved -> Html/[1414]東和-紡織纖維-上市.html
 Visit: http://localhost/IT//Html/[1414]東和-紡織纖維-上市.html
【craw_stock】 craw_stock stock_number!!!!!!!!! -->1435saved -> Html/[1416]廣豐-其他-上市.html
 Visit: http://localhost/IT//Html/[1416]廣豐-其他-上市.html

saved -> Html/[1417]嘉裕-紡織纖維-上市.html
 Visit: http://localhost/IT//Html/[1417]嘉裕-紡織纖維-上市.html
【craw_stock】 craw_stock stock_number!!!!!!!!! -->1436
【craw_stock】 craw_stock stock_number!!!!!!!!! -->1437
saved -> Html/[1418]東華-紡織纖維-上市.html
 Visit: http://localhost/IT

【craw_stock】 craw_stock stock_number!!!!!!!!! -->1526
saved -> Html/[1517]利奇-電機機械-上市.html
 Visit: http://localhost/IT//Html/[1517]利奇-電機機械-上市.html


【craw_stock】 craw_stock stock_number!!!!!!!!! -->1527


【craw_stock】 craw_stock stock_number!!!!!!!!! -->1528
saved -> Html/[1519]華城-電機機械-上市.html
 Visit: http://localhost/IT//Html/[1519]華城-電機機械-上市.html
【craw_stock】 craw_stock stock_number!!!!!!!!! -->1529
saved -> Html/[1521]大億-汽車工業-上市.html
 Visit: http://localhost/IT//Html/[1521]大億-汽車工業-上市.html
【craw_stock】 craw_stock stock_number!!!!!!!!! -->1530
saved -> Html/[1524]耿鼎-汽車工業-上市.html
 Visit: http://localhost/IT//Html/[1524]耿鼎-汽車工業-上市.html
saved -> Html/[1522]堤維西-汽車工業-上市.html
 Visit: http://localhost/IT//Html/[1522]堤維西-汽車工業-上市.html

【craw_stock】 craw_stock stock_number!!!!!!!!! -->1531
saved -> Html/[1525]江申-汽車工業-上市.html
 Visit: http://localhost/IT//Html/[1525]江申-汽車工業-上市.html
【craw_stock】 craw_stock stock_number!!!!!!!!! -->1532
saved -> Html/[1526]日馳-電機機械-上市.html
 Visit: http://localhost/IT

【craw_stock】 craw_stock stock_number!!!!!!!!! -->1727
saved -> Html/[1718]中纖-化學工業-上市.html
 Visit: http://localhost/IT//Html/[1718]中纖-化學工業-上市.html

【craw_stock】 craw_stock stock_number!!!!!!!!! -->1730
saved -> Html/[1720]生達-生技醫療業-上市.html
 Visit: http://localhost/IT//Html/[1720]生達-生技醫療業-上市.html

saved -> Html/[1721]三晃-化學工業-上市.html
 Visit: http://localhost/IT//Html/[1721]三晃-化學工業-上市.html
【craw_stock】 craw_stock stock_number!!!!!!!!! -->1731

【craw_stock】 craw_stock stock_number!!!!!!!!! -->1732
saved -> Html/[1722]台肥-化學工業-上市.html
 Visit: http://localhost/IT//Html/[1722]台肥-化學工業-上市.html



【craw_stock】 craw_stock stock_number!!!!!!!!! -->1733
saved -> Html/[1723]中碳-化學工業-上市.html
 Visit: http://localhost/IT//Html/[1723]中碳-化學工業-上市.html
saved -> Html/[1725]元禎-化學工業-上市.html
 Visit: http://localhost/IT//Html/[1725]元禎-化學工業-上市.html
【craw_stock】 craw_stock stock_number!!!!!!!!! -->1734

saved -> Html/[1726]永記-化學工業-上市.html
 Visit: http://localhost/IT//Html/[1726]永記-化學工業-上市.html
【craw_stock】 craw_stock

saved -> Html/[2029]盛餘-鋼鐵工業-上市.html
 Visit: http://localhost/IT//Html/[2029]盛餘-鋼鐵工業-上市.html
【craw_stock】 craw_stock stock_number!!!!!!!!! -->2049
saved -> Html/[2030]彰源-鋼鐵工業-上市.html
 Visit: http://localhost/IT//Html/[2030]彰源-鋼鐵工業-上市.html
【craw_stock】 craw_stock stock_number!!!!!!!!! -->2059

【craw_stock】 craw_stock stock_number!!!!!!!!! -->2062
saved -> Html/[2031]新光鋼-鋼鐵工業-上市.html
 Visit: http://localhost/IT//Html/[2031]新光鋼-鋼鐵工業-上市.html


【craw_stock】 craw_stock stock_number!!!!!!!!! -->2069
saved -> Html/[2032]新鋼-鋼鐵工業-上市.html
 Visit: http://localhost/IT//Html/[2032]新鋼-鋼鐵工業-上市.html
saved -> Html/[2033]佳大-鋼鐵工業-上市.html
 Visit: http://localhost/IT//Html/[2033]佳大-鋼鐵工業-上市.html
【craw_stock】 craw_stock stock_number!!!!!!!!! -->2072
saved -> Html/[2034]允強-鋼鐵工業-上市.html
 Visit: http://localhost/IT//Html/[2034]允強-鋼鐵工業-上市.html

【craw_stock】 craw_stock stock_number!!!!!!!!! -->2101


saved -> Html/[2038]海光-鋼鐵工業-上市.html
 Visit: http://localhost/IT//Html/[2038]海光-鋼鐵工業-上市.html
【craw_stock】 craw_stock 

...

saved -> Html/[6761]穩得-電子零組件業-上櫃.html
 Visit: http://localhost/IT//Html/[6761]穩得-電子零組件業-上櫃.html
【craw_stock】 craw_stock stock_number!!!!!!!!! -->6791
saved -> Html/[6762]達亞-生技醫療業-上櫃.html
 Visit: http://localhost/IT//Html/[6762]達亞-生技醫療業-上櫃.html
saved -> Html/[6763]綠界科技-數位雲端-上櫃.html
 Visit: http://localhost/IT//Html/[6763]綠界科技-數位雲端-上櫃.html
【craw_stock】 craw_stock stock_number!!!!!!!!! -->6803
saved -> Html/[6767]台微醫-生技醫療業-上櫃.html
 Visit: http://localhost/IT//Html/[6767]台微醫-生技醫療業-上櫃.html
【craw_stock】 craw_stock stock_number!!!!!!!!! -->6804


saved -> Html/[6785]昱展新藥-生技醫療業-上櫃.html
 Visit: http://localhost/IT//Html/[6785]昱展新藥-生技醫療業-上櫃.html
【craw_stock】 craw_stock stock_number!!!!!!!!! -->6811


【craw_stock】 craw_stock stock_number!!!!!!!!! -->6821saved -> Html/[6788]華景電-半導體業-上櫃.html
 Visit: http://localhost/IT//Html/[6788]華景電-半導體業-上櫃.html


saved -> Html/[6791]虎門科技-資訊服務業-上櫃.html
 Visit: http://localhost/IT//Html/[6791]虎門科技-資訊服務業-上櫃.html
【craw_stock】 craw_stock stock_number!!!!!!!!! -->6823

saved -> Html/[6971]惠民實業-綠能環保-上櫃.html
 Visit: http://localhost/IT//Html/[6971]惠民實業-綠能環保-上櫃.html
【craw_stock】 craw_stock stock_number!!!!!!!!! -->7402



saved -> Html/[6983]華洋精機-其他電子業-上櫃.html
 Visit: http://localhost/IT//Html/[6983]華洋精機-其他電子業-上櫃.html
【craw_stock】 craw_stock stock_number!!!!!!!!! -->7547
saved -> Html/[6982]大井泵浦-電機機械-上櫃.html
 Visit: http://localhost/IT//Html/[6982]大井泵浦-電機機械-上櫃.html

【craw_stock】 craw_stock stock_number!!!!!!!!! -->7556

【craw_stock】 craw_stock stock_number!!!!!!!!! -->7584
saved -> Html/[6996]力領科技-半導體業-上櫃.html
 Visit: http://localhost/IT//Html/[6996]力領科技-半導體業-上櫃.html

【craw_stock】 craw_stock stock_number!!!!!!!!! -->7642



saved -> Html/[6997]博弘-數位雲端-上櫃.html
 Visit: http://localhost/IT//Html/[6997]博弘-數位雲端-上櫃.html
saved -> Html/[7402]邑錡-光電業-上櫃.html
 Visit: http://localhost/IT//Html/[7402]邑錡-光電業-上櫃.html
【craw_stock】 craw_stock stock_number!!!!!!!!! -->7703

【craw_stock】 craw_stock stock_number!!!!!!!!! -->7704
saved -> Html/[7547]碩網-數位雲端-上櫃.html
 Visit: 

【craw_stock】 craw_stock stock_number!!!!!!!!! -->8054


saved -> Html/[8044]網家-數位雲端-上櫃.html
 Visit: http://localhost/IT//Html/[8044]網家-數位雲端-上櫃.html
saved -> Html/[8047]星雲-其他電子業-上櫃.html
 Visit: http://localhost/IT//Html/[8047]星雲-其他電子業-上櫃.html
【craw_stock】 craw_stock stock_number!!!!!!!!! -->8059

【craw_stock】 craw_stock stock_number!!!!!!!!! -->8064
saved -> Html/[8048]德勝-通信網路業-上櫃.html
 Visit: http://localhost/IT//Html/[8048]德勝-通信網路業-上櫃.html
saved -> Html/[8049]晶采-光電業-上櫃.html
 Visit: http://localhost/IT//Html/[8049]晶采-光電業-上櫃.html
【craw_stock】 craw_stock stock_number!!!!!!!!! -->8066

【craw_stock】 craw_stock stock_number!!!!!!!!! -->8067
saved -> Html/[8050]廣積-電腦及週邊設備業-上櫃.html
 Visit: http://localhost/IT//Html/[8050]廣積-電腦及週邊設備業-上櫃.html


saved -> Html/[8054]安國-半導體業-上櫃.html
 Visit: http://localhost/IT//Html/[8054]安國-半導體業-上櫃.html
【craw_stock】 craw_stock stock_number!!!!!!!!! -->8068
saved -> Html/[8059]凱碩-通信網路業-上櫃.html
 Visit: http://localhost/IT//Html/[8059]凱碩-通信網路業-上櫃.html
【craw_stock】 c

【craw_stock】 craw_stock stock_number!!!!!!!!! -->8409
saved -> Html/[8358]金居-電子零組件業-上櫃.html
 Visit: http://localhost/IT//Html/[8358]金居-電子零組件業-上櫃.html


saved -> Html/[8383]千附-半導體業-上櫃.html
 Visit: http://localhost/IT//Html/[8383]千附-半導體業-上櫃.html
【craw_stock】 craw_stock stock_number!!!!!!!!! -->8410

saved -> Html/[8390]金益鼎-綠能環保-上櫃.html
 Visit: http://localhost/IT//Html/[8390]金益鼎-綠能環保-上櫃.html
【craw_stock】 craw_stock stock_number!!!!!!!!! -->8415
【craw_stock】 craw_stock stock_number!!!!!!!!! -->8416
saved -> Html/[8401]白紗科-其他-上櫃.html
 Visit: http://localhost/IT//Html/[8401]白紗科-其他-上櫃.html
saved -> Html/[8403]盛弘-生技醫療業-上櫃.html
 Visit: http://localhost/IT//Html/[8403]盛弘-生技醫療業-上櫃.html
【craw_stock】 craw_stock stock_number!!!!!!!!! -->8421

【craw_stock】 craw_stock stock_number!!!!!!!!! -->8423
saved -> Html/[8409]商之器-生技醫療業-上櫃.html
 Visit: http://localhost/IT//Html/[8409]商之器-生技醫療業-上櫃.html

saved -> Html/[8410]森田-電腦及週邊設備業-上櫃.html
 Visit: http://localhost/IT//Html/[8410]森田-電腦及週邊設備業-上櫃.html
【craw_sto

2026-06-10 00:00:00
2026-06-08 00:00:00
2026-06-11 00:00:00
2026-06-08 00:00:00
2026-06-12 00:00:00
2026-06-08 00:00:00
2026-06-15 00:00:00
2026-06-08 00:00:00
2026-06-16 00:00:00
2026-06-08 00:00:00
2026-06-17 00:00:00
2026-06-08 00:00:00
2026-06-18 00:00:00
2026-06-08 00:00:00
2026-06-22 00:00:00
2026-06-08 00:00:00
2026-06-23 00:00:00
2026-06-08 00:00:00
2026-06-24 00:00:00
2026-06-08 00:00:00
2026-06-25 00:00:00
2026-06-08 00:00:00
2026-06-26 00:00:00
2026-06-08 00:00:00
All stocks processed.
start time 2026/06/27 13:34:42
end time   2026/06/27 14:12:19
elapsed    0:37:37.110677


In [8]:
%run Untitled.ipynb

今天是： 星期六
2026-06-26
2026-06-26
Base 資料 ['2026-06-12', '2026-06-13', '2026-06-14', '2026-06-15', '2026-06-16', '2026-06-17', '2026-06-18', '2026-06-19', '2026-06-20', '2026-06-21', '2026-06-22', '2026-06-23', '2026-06-24', '2026-06-25', '2026-06-26']
join 新資料日 2026-06-26
error----->D:\Project\Jupyter\Stock\Main\Result\DailyFiles\2026-06-13.xlsx
error----->D:\Project\Jupyter\Stock\Main\Result\DailyFiles\2026-06-14.xlsx
error----->D:\Project\Jupyter\Stock\Main\Result\DailyFiles\2026-06-19.xlsx
error----->D:\Project\Jupyter\Stock\Main\Result\DailyFiles\2026-06-20.xlsx
error----->D:\Project\Jupyter\Stock\Main\Result\DailyFiles\2026-06-21.xlsx
2026-06-26
Result/checksun_a
16
Result/checksun_b1
345
Result/checksun_b2
7
Result/checksun_c
734
Result/checksun_d1
206
Result/checksun_d2
127
Result/checksun_e
396
Result/checksun_x
41
Result/checksun_y
45
361
AI伺服器
AI應用服務
AI晶片
AI電力基建
PCB載板
光學
光通訊
光電
其他
其他電子
功率元件
化學
半導體
半導體材料
半導體設備
封裝測試
導線架
居家生活
工業自動化
工業電腦
數位雲端
文化創意
機器人
橡膠
水泥
汽車
油電燃氣
營建
玻璃
玻璃陶瓷
生技
生技